# CheckThat Task 3 — Stage 2 Claim-Aware Evidence Extraction

**Version:** `ROLE_RISK_v1`

This notebook is a replacement for your previous Stage 2 notebook. It is aligned with the newer four-stage pipeline design:

- Stage 2 runs only on Stage 1 `PROCESS` sources.
- Stage 2 does **not** use the original/final rating.
- Stage 2 does **not** use the reference review article.
- Python restores exact identity fields: `id`, `claim`, `url`, `file_name`.
- The model returns extraction-only fields, including `evidence_role` and `risk_flags`.
- Guardrails prevent claim-origin, satire, fake-origin, visual-origin, and weak context sources from being misused downstream.
- The notebook supports resume, local + Drive saving, smoke testing, and parallel API calls.

**Main output:**

```text
outputs/stage2/stage2_extract_TARGET10.json
```

Change `DATASET_TAG` to run another split/sample, for example `100_test`.

## 1. Install and authenticate

Run this first in Colab. If Vertex rejects `gemini-3-flash-preview`, switch `STAGE2_MODEL` in the config cell to a model available in your project, for example `gemini-2.5-flash`.

In [1]:
!pip install -q -U google-genai

from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")

import os
from google import genai
from google.genai import types

PROJECT_ID = "clef-checkthat"
LOCATION = "global"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

print("Vertex Gemini client ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 689.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 791.9/791.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 16.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.52.0 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.0.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.0.0 which is incompatible.
Mounted at /content/drive
Vertex Gemini client ready.


## 2. Configuration

This cell is the main place to edit paths, dataset tag, model, and concurrency.

Recommended workflow:

1. Keep `TEST_ONE_SOURCE = True` for a smoke test.
2. After the smoke test looks good, set `TEST_ONE_SOURCE = False` and run the full extraction cell.
3. If you hit rate limits, reduce `MAX_WORKERS` to `4`, `2`, or `1`.

In [2]:
from pathlib import Path

DRIVE_BASE = Path("/content/drive/MyDrive/CheckThat_Task3_Dataset")

# Change this to run another split/sample.
# Examples: "TARGET10", "100_test", "train", "dev"
DATASET_TAG = "test_dataset"

# Stage 0 input. This assumes your Stage 0 file naming convention is unchanged.
STAGE0_IN = DRIVE_BASE / f"outputs/stage0/flattened_sources_{DATASET_TAG}.json"

# Stage 1 input candidates.
# The first candidate matches the new pipeline design; the second supports your older notebook naming.
STAGE1_IN_CANDIDATES = [
    DRIVE_BASE / f"outputs/stage1/stage1_sources_{DATASET_TAG}.json",
    DRIVE_BASE / f"outputs/stage1/stage1_api_triage_{DATASET_TAG}.json",
]

LOCAL_STAGE2_DIR = Path("/content/outputs/stage2")
LOCAL_STAGE2_DIR.mkdir(parents=True, exist_ok=True)
STAGE2_OUT = LOCAL_STAGE2_DIR / f"stage2_extract_{DATASET_TAG}.json"

DRIVE_STAGE2_DIR = DRIVE_BASE / "outputs/stage2"
DRIVE_STAGE2_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_STAGE2_OUT = DRIVE_STAGE2_DIR / STAGE2_OUT.name

# Recommended Stage 2 model. If unavailable, try "gemini-2.5-flash".
STAGE2_MODEL = "gemini-3-flash-preview"
STAGE2_TEMPERATURE = 0.0
STAGE2_MAX_OUTPUT_TOKENS = 20000

# Gemini 3 supports thinking_level. If rejected, the call function retries without it.
STAGE2_THINKING_LEVEL = "medium"  # "low", "medium", "high", or None

MAX_RETRIES = 5
MAX_STAGE2_SOURCE_CHARS = 30000

# Parallel extraction settings.
# Set MAX_WORKERS = 1 for fully sequential debugging.
MAX_WORKERS = 8
SAVE_AFTER_EVERY_COMPLETED = 20

# Optional API call cap for debugging. Set to None for full run.
MAX_API_CALLS = None

TEST_ONE_SOURCE = False
INSPECT_ID = "30350"

print("Dataset tag:", DATASET_TAG)
print("Stage 0 input:", STAGE0_IN)
print("Stage 1 candidates:")
for p in STAGE1_IN_CANDIDATES:
    print(" -", p)
print("Stage 2 local output:", STAGE2_OUT)
print("Stage 2 Drive output:", DRIVE_STAGE2_OUT)
print("Model:", STAGE2_MODEL)
print("MAX_WORKERS:", MAX_WORKERS)

Dataset tag: test_dataset
Stage 0 input: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage0/flattened_sources_test_dataset.json
Stage 1 candidates:
 - /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage1/stage1_sources_test_dataset.json
 - /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage1/stage1_api_triage_test_dataset.json
Stage 2 local output: /content/outputs/stage2/stage2_extract_test_dataset.json
Stage 2 Drive output: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage2/stage2_extract_test_dataset.json
Model: gemini-3-flash-preview
MAX_WORKERS: 8


## 3. Utility functions

In [3]:
import json
import re
import time
import random
import shutil
from datetime import datetime, timezone
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed


def load_json(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    tmp.replace(path)


def copy_to_drive(local_path, drive_path):
    drive_path = Path(drive_path)
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local_path, drive_path)


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def source_key(x):
    return f'{str(x["id"])}||{x["file_name"]}||{x["url"]}'


def find_existing_path(candidates, label):
    for path in candidates:
        if Path(path).exists():
            print(f"Using {label}:", path)
            return Path(path)
    raise FileNotFoundError(
        f"Could not find {label}. Tried:\n" + "\n".join(str(p) for p in candidates)
    )


def get_response_text(response):
    text = getattr(response, "text", None)
    if text:
        return text.strip()

    chunks = []
    for cand in getattr(response, "candidates", None) or []:
        content = getattr(cand, "content", None)
        parts = getattr(content, "parts", None) if content else None
        for part in parts or []:
            part_text = getattr(part, "text", None)
            if part_text:
                chunks.append(part_text)
    return "\n".join(chunks).strip()


def extract_json_response(response):
    parsed = getattr(response, "parsed", None)
    if parsed:
        if isinstance(parsed, dict):
            return parsed
        try:
            return json.loads(json.dumps(parsed))
        except Exception:
            pass

    raw = get_response_text(response)
    if not raw:
        raise ValueError("Empty model response")

    raw = re.sub(r"^```(?:json)?\s*", "", raw.strip(), flags=re.IGNORECASE)
    raw = re.sub(r"\s*```$", "", raw).strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start = raw.find("{")
        end = raw.rfind("}")
        if start >= 0 and end > start:
            return json.loads(raw[start:end + 1])
        raise


def response_usage_summary(response):
    usage = getattr(response, "usage_metadata", None)
    if usage is None:
        return {}
    return {
        "prompt_token_count": getattr(usage, "prompt_token_count", None),
        "candidates_token_count": getattr(usage, "candidates_token_count", None),
        "thoughts_token_count": getattr(usage, "thoughts_token_count", None),
        "total_token_count": getattr(usage, "total_token_count", None),
    }


def clean_space(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()


def trim_for_stage2(text):
    text = text or ""
    if len(text) <= MAX_STAGE2_SOURCE_CHARS:
        return text
    return text[:MAX_STAGE2_SOURCE_CHARS] + "\n\n[TRUNCATED FOR STAGE 2 API CALL]"


def clamp_int(value, lo=0, hi=5, default=0):
    try:
        value = int(value)
    except Exception:
        return default
    return max(lo, min(hi, value))

## 4. Load Stage 0 + Stage 1 and prepare Stage 2 inputs

Stage 2 uses only sources that Stage 1 marked as `PROCESS`. It still does **not** trust the model for identity fields.

In [4]:
assert STAGE0_IN.exists(), f"Missing Stage 0 file: {STAGE0_IN}"
STAGE1_IN = find_existing_path(STAGE1_IN_CANDIDATES, "Stage 1 input")

flattened_sources = load_json(STAGE0_IN)
stage1_outputs = load_json(STAGE1_IN)

stage1_by_key = {source_key(x): x for x in stage1_outputs}

stage2_inputs = []
missing_stage1 = []
skipped = []

for source in flattened_sources:
    key = source_key(source)
    triage = stage1_by_key.get(key)

    if not triage:
        missing_stage1.append(source)
        continue

    if triage.get("stage1_decision") != "PROCESS":
        skipped.append(source)
        continue

    item = dict(source)
    item["id"] = str(item["id"])
    item["source_text"] = trim_for_stage2(item.get("source_text", ""))
    item["stage1_decision"] = triage.get("stage1_decision")
    item["stage1_confidence"] = triage.get("stage1_confidence")
    item["stage1_reason"] = triage.get("stage1_reason")
    stage2_inputs.append(item)

print("Flattened Stage 0 sources:", len(flattened_sources))
print("Stage 1 outputs:", len(stage1_outputs))
print("Stage 1 SKIP sources:", len(skipped))
print("Missing Stage 1 records:", len(missing_stage1))
print("Stage 2 PROCESS inputs:", len(stage2_inputs))

print("\nFirst few Stage 2 inputs:")
for x in stage2_inputs[:10]:
    print("-", x["id"], x["file_name"], x["url"][:90])

Using Stage 1 input: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage1/stage1_api_triage_test_dataset.json
Flattened Stage 0 sources: 8979
Stage 1 outputs: 8979
Stage 1 SKIP sources: 3238
Missing Stage 1 records: 0
Stage 2 PROCESS inputs: 5741

First few Stage 2 inputs:
- 30354_exclaim 30354_2.json https://www.cnn.com/TRANSCRIPTS/1606/21/sitroom.02.html
- 30354_exclaim 30354_3.json https://www.politifact.com/truth-o-meter/statements/2016/jun/21/hillary-clinton/yep-donald
- 30354_exclaim 30354_4.json https://www.politifact.com/truth-o-meter/statements/2015/sep/21/carly-fiorina/trumps-four-
- 30354_exclaim 30354_5.json https://money.cnn.com/2015/08/31/news/companies/donald-trump-bankruptcy/
- 30354_exclaim 30354_6.json https://abcnews.go.com/Politics/donald-trump-filed-bankruptcy-times/story?id=13419250
- 30354_exclaim 30354_7.json https://www.forbes.com/sites/debtwire/2015/08/18/a-trip-down-donald-trumps-bankruptcy-memo
- 30354_exclaim 30354_8.json https://www.wsj.com/articl

## 5. Stage 2 labels and response schema

This version uses the newer `evidence_role` field as the main bridge into Stage 3.

In [5]:
ALLOWED_SOURCE_RELATIONS = {
    "direct_evidence",
    "claim_origin",
    "useful_context",
    "off_topic",
    "no_content",
}

ALLOWED_EVIDENCE_ROLES = {
    "core_support",
    "core_refutation",
    "claim_origin",
    "visual_origin",
    "visual_misattribution",
    "image_manipulation",
    "satirical_origin",
    "fake_news_origin",
    "temporal_refutation",
    "rating_caveat",
    "statistical_evidence",
    "legal_or_definition_context",
    "scientific_context",
    "background_context",
    "off_topic",
    "no_content",
}

ALLOWED_STANCES = {
    "supports_claim",
    "refutes_claim",
    "refutes_supporting_evidence",
    "qualifies_claim",
    "satire",
    "context_only",
    "not_applicable",
    "unclear",
}

ALLOWED_SNIPPET_TYPES = {
    "evidence",
    "claim_origin",
    "visual_origin",
    "satire_context",
    "fake_origin_context",
    "context",
}

ALLOWED_SNIPPET_STANCES = ALLOWED_STANCES - {"not_applicable"}

SPECIAL_PROMOTION_ROLES = {
    "visual_origin",
    "visual_misattribution",
    "image_manipulation",
    "satirical_origin",
    "fake_news_origin",
    "temporal_refutation",
    "rating_caveat",
    "statistical_evidence",
}

DIRECT_EVIDENCE_ROLES = {
    "core_support",
    "core_refutation",
    "visual_origin",
    "visual_misattribution",
    "image_manipulation",
    "temporal_refutation",
    "rating_caveat",
    "statistical_evidence",
}

CONTEXT_ROLES = {
    "legal_or_definition_context",
    "scientific_context",
    "background_context",
}

# Risk flags are intentionally free-form strings because useful flags often need details, e.g.
# "predicate_mismatch: claim says played, evidence says visited".
RECOMMENDED_RISK_FLAG_PREFIXES = [
    "entity_mismatch",
    "location_mismatch",
    "date_mismatch",
    "predicate_mismatch",
    "number_unit_mismatch",
    "trial_vs_approval_mismatch",
    "visited_vs_played_mismatch",
    "satire_vs_factual_mismatch",
    "image_misattribution",
    "visual_origin_only",
    "image_manipulation",
    "claim_origin_not_proof",
    "fake_news_origin",
    "temporal_mismatch",
    "rating_caveat",
    "statistical_exactness_needed",
    "boilerplate_or_sidebar",
    "weak_context_only",
    "off_topic_or_boilerplate",
    "no_readable_content",
]

STAGE2_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "source_relation": {"type": "string", "enum": sorted(ALLOWED_SOURCE_RELATIONS)},
        "evidence_role": {"type": "string", "enum": sorted(ALLOWED_EVIDENCE_ROLES)},
        "claim_match_score": {"type": "integer"},
        "stance_to_claim": {"type": "string", "enum": sorted(ALLOWED_STANCES)},
        "keep_for_writing": {"type": "boolean"},
        "keep_for_coverage": {"type": "boolean"},
        "why": {"type": "string"},
        "snippets": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "snippet_type": {"type": "string", "enum": sorted(ALLOWED_SNIPPET_TYPES)},
                    "text": {"type": "string"},
                    "stance": {"type": "string", "enum": sorted(ALLOWED_SNIPPET_STANCES)},
                    "snippet_score": {"type": "integer"},
                },
                "required": ["snippet_type", "text", "stance", "snippet_score"],
            },
        },
        "safe_source_fact": {"type": "string"},
        "risk_flags": {
            "type": "array",
            "items": {"type": "string"},
        },
    },
    "required": [
        "source_relation",
        "evidence_role",
        "claim_match_score",
        "stance_to_claim",
        "keep_for_writing",
        "keep_for_coverage",
        "why",
        "snippets",
        "safe_source_fact",
        "risk_flags",
    ],
}

print("Allowed evidence roles:", len(ALLOWED_EVIDENCE_ROLES))
print("Special promotion roles:", sorted(SPECIAL_PROMOTION_ROLES))

Allowed evidence roles: 16
Special promotion roles: ['fake_news_origin', 'image_manipulation', 'rating_caveat', 'satirical_origin', 'statistical_evidence', 'temporal_refutation', 'visual_misattribution', 'visual_origin']


## 6. Prompt builder

The prompt asks the model to extract evidence, not to write the article and not to decide the final verdict.

In [6]:
def build_stage2_prompt(source):
    risk_prefixes = "\n".join(f"- {x}" for x in RECOMMENDED_RISK_FLAG_PREFIXES)
    roles = "\n".join(f"- {x}" for x in sorted(ALLOWED_EVIDENCE_ROLES))

    return f"""You are extracting claim-relevant evidence from one scraped source for a fact-check article generation pipeline.

Use only the source text provided.
Do not use outside knowledge.
Do not decide the final verdict.
Do not use the final rating or original rating.
Do not use any reference review article.
Do not return the claim ID, URL, or file name. Python will add them exactly.

Your job:
1. Decide whether this source helps check the claim.
2. Assign an evidence_role that tells Stage 3 how to use this source.
3. Extract short citation-ready snippets/facts from the main source content only.
4. Flag mismatches between the claim and evidence.
5. Avoid boilerplate, menus, related-story boxes, ads, donation blocks, comments, and unrelated sidebar content.
6. Keep evidence small and precise.

Allowed source_relation values:
- direct_evidence: source gives factual evidence for/against the claim, or decisive provenance evidence for visual/image/video/screenshot claims.
- claim_origin: source is where the claim was made/spread, including fake-news/parody/satire origins. This is not proof that the claim is true.
- useful_context: helpful context that may explain the claim, rating nuance, legal/scientific/statistical background, or definitions, but is not direct proof by itself.
- off_topic: readable but not useful for checking the claim.
- no_content: no readable useful content.

Required evidence_role values:
{roles}

Role guidance:
- core_support: directly supports the factual claim.
- core_refutation: directly refutes the factual claim.
- claim_origin: shows who made/spread the claim, but does not prove it.
- visual_origin: identifies the real origin/provenance of an image/video/screenshot.
- visual_misattribution: shows that a visual was reused/misattributed to the wrong event/person/place/date.
- image_manipulation: shows an image/screenshot/video was edited, fabricated, or manipulated.
- satirical_origin: source is satire/parody/fiction or identifies the claim as satire/parody.
- fake_news_origin: source is a fake-news/parody/fabricated story origin.
- temporal_refutation: timing/date evidence refutes or qualifies the claim.
- rating_caveat: important nuance needed for mostly_true, half_true, misleading, or similar ratings.
- statistical_evidence: exact number, statistic, denominator, unit, or measurement evidence.
- legal_or_definition_context: law, policy, official definition, or conceptual definition context.
- scientific_context: medical/scientific/technical context.
- background_context: useful background only.
- off_topic/no_content: not useful or unreadable.

Allowed stance_to_claim values:
- supports_claim
- refutes_claim
- refutes_supporting_evidence
- qualifies_claim
- satire
- context_only
- not_applicable
- unclear

Claim match score:
5 = exact direct support/refutation or decisive provenance evidence
4 = strong evidence for a central part of the claim
3 = claim origin or necessary context
2 = weak context
1 = barely useful
0 = off-topic or no readable content

Keep-for-writing rule:
- keep_for_writing = true only if claim_match_score >= 3, source_relation is direct_evidence/claim_origin/useful_context, and at least one useful snippet exists.
- Exception: if the role is visual_origin, visual_misattribution, image_manipulation, satirical_origin, fake_news_origin, temporal_refutation, rating_caveat, or statistical_evidence and score >= 4, keep_for_writing should be true.

Critical rules:
- A claim-origin source must not be treated as proof. Use source_relation=claim_origin and stance_to_claim=context_only or unclear.
- A satire/parody source must not be labeled supports_claim. Use evidence_role=satirical_origin and stance_to_claim=satire or context_only.
- A fake-news/parody-origin source must not be labeled supports_claim. Use evidence_role=fake_news_origin and source_relation=claim_origin.
- A source that only proves a post/tweet/headline exists does not support the real-world claim.
- If the source identifies the real origin of a reused image/video/screenshot, use visual_origin or visual_misattribution.
- If the claim has an exact number, do not support it unless the exact number, unit, denominator, and framing match.
- Useful context is not direct evidence. Do not label weak background as supports_claim/refutes_claim.
- If keep_for_writing is true, include at least one useful snippet.
- If keep_for_coverage is true, include a safe_source_fact.

Risk flags:
Use risk_flags to capture mismatches or usage warnings. Recommended prefixes include:
{risk_prefixes}

Return strict JSON only matching the schema.

Input:
Claim ID: {source["id"]}
Claim: {source.get("claim", "")}
URL: {source["url"]}
File name: {source["file_name"]}
Stage 1 decision: {source.get("stage1_decision", "")}
Stage 1 reason: {source.get("stage1_reason", "")}

Source text:
<<<SOURCE_TEXT_START
{source.get("source_text", "")}
SOURCE_TEXT_END>>>
"""

## 7. Validation, normalization, and deterministic guardrails

These functions prevent bad label combinations from leaking into Stage 3.

In [7]:
def validate_stage2_model_output(obj):
    errors = []

    if not isinstance(obj, dict):
        return ["Output is not a JSON object"]

    missing = set(STAGE2_RESPONSE_SCHEMA["required"]) - set(obj.keys())
    if missing:
        errors.append(f"Missing required keys: {sorted(missing)}")

    if obj.get("source_relation") not in ALLOWED_SOURCE_RELATIONS:
        errors.append("Invalid source_relation")

    if obj.get("evidence_role") not in ALLOWED_EVIDENCE_ROLES:
        errors.append("Invalid evidence_role")

    score = obj.get("claim_match_score")
    if not isinstance(score, int) or not (0 <= score <= 5):
        errors.append("claim_match_score must be an integer from 0 to 5")

    if obj.get("stance_to_claim") not in ALLOWED_STANCES:
        errors.append("Invalid stance_to_claim")

    if not isinstance(obj.get("keep_for_writing"), bool):
        errors.append("keep_for_writing must be boolean")

    if not isinstance(obj.get("keep_for_coverage"), bool):
        errors.append("keep_for_coverage must be boolean")

    if not isinstance(obj.get("why"), str):
        errors.append("why must be a string")

    snippets = obj.get("snippets")
    if not isinstance(snippets, list):
        errors.append("snippets must be a list")
        snippets = []

    for i, snip in enumerate(snippets):
        if not isinstance(snip, dict):
            errors.append(f"snippet {i} is not an object")
            continue

        if snip.get("snippet_type") not in ALLOWED_SNIPPET_TYPES:
            errors.append(f"snippet {i} has invalid snippet_type")

        if not isinstance(snip.get("text"), str) or not snip.get("text", "").strip():
            errors.append(f"snippet {i} text is empty")

        if snip.get("stance") not in ALLOWED_SNIPPET_STANCES:
            errors.append(f"snippet {i} has invalid stance")

        ss = snip.get("snippet_score")
        if not isinstance(ss, int) or not (0 <= ss <= 5):
            errors.append(f"snippet {i} snippet_score must be 0 to 5")

    if obj.get("keep_for_writing") is True and not snippets:
        errors.append("keep_for_writing=true requires at least one snippet")

    if obj.get("keep_for_coverage") is True and not clean_space(obj.get("safe_source_fact", "")):
        errors.append("keep_for_coverage=true requires safe_source_fact")

    if not isinstance(obj.get("risk_flags"), list):
        errors.append("risk_flags must be a list")
    else:
        for flag in obj.get("risk_flags", []):
            if not isinstance(flag, str):
                errors.append("risk_flags must contain strings only")
                break

    return errors


def _add_flag(risk_flags, value):
    value = clean_space(value)
    if value and value not in risk_flags:
        risk_flags.append(value)


def normalize_snippets(raw_snippets):
    snippets = []
    seen = set()

    for snip in raw_snippets or []:
        if not isinstance(snip, dict):
            continue
        text = clean_space(snip.get("text", ""))
        if not text:
            continue
        # Keep snippets short enough to be citation-ready for Stage 3/4.
        text = text[:700]
        key = text.lower()
        if key in seen:
            continue
        seen.add(key)

        snippet_type = snip.get("snippet_type")
        if snippet_type not in ALLOWED_SNIPPET_TYPES:
            snippet_type = "context"

        stance = snip.get("stance")
        if stance not in ALLOWED_SNIPPET_STANCES:
            stance = "unclear"

        snippets.append({
            "snippet_type": snippet_type,
            "text": text,
            "stance": stance,
            "snippet_score": clamp_int(snip.get("snippet_score", 0)),
        })

    return snippets


def normalize_risk_flags(raw_flags):
    flags = []
    for flag in raw_flags or []:
        flag = clean_space(flag)[:250]
        if flag and flag not in flags:
            flags.append(flag)
    return flags


def apply_stage2_guardrails(relation, role, score, stance, snippets, safe_source_fact, risk_flags):
    """Deterministic cleanup so known bad combinations cannot leak into Stage 3."""

    score = clamp_int(score)

    # Role determines relation in the clearest high-risk cases.
    if role == "off_topic":
        relation = "off_topic"
    if role == "no_content":
        relation = "no_content"

    # Off-topic / no-content rows should never be kept.
    if relation in {"off_topic", "no_content"}:
        role = "off_topic" if relation == "off_topic" else "no_content"
        score = 0
        stance = "not_applicable" if relation == "off_topic" else "unclear"
        snippets = []
        safe_source_fact = ""
        _add_flag(risk_flags, "off_topic_or_boilerplate" if relation == "off_topic" else "no_readable_content")
        return relation, role, score, stance, snippets, safe_source_fact, risk_flags

    # Ordinary claim-origin records show the claim was made/spread, not that it is true.
    # Do not cap satirical/fake origins here because the pipeline explicitly promotes
    # high-value satire/fake-origin evidence when score >= 4.
    if role == "claim_origin" or (relation == "claim_origin" and role not in {"satirical_origin", "fake_news_origin"}):
        relation = "claim_origin"
        role = "claim_origin"
        score = min(score, 3)
        if stance in {"supports_claim", "refutes_claim", "refutes_supporting_evidence", "qualifies_claim"}:
            stance = "context_only"
        _add_flag(risk_flags, "claim_origin_not_proof")
        for snip in snippets:
            snip["snippet_type"] = "claim_origin"
            if snip.get("stance") in {"supports_claim", "refutes_claim", "refutes_supporting_evidence", "qualifies_claim"}:
                snip["stance"] = "context_only"
            snip["snippet_score"] = min(clamp_int(snip.get("snippet_score", 0)), 3)

    # Satire/parody/fake origins are important but must not support the real-world claim.
    if role == "satirical_origin":
        relation = "claim_origin"
        score = max(score, 3)
        if stance == "supports_claim":
            stance = "satire"
        if stance not in {"satire", "context_only", "unclear"}:
            stance = "satire"
        _add_flag(risk_flags, "satire_vs_factual_mismatch")
        for snip in snippets:
            snip["snippet_type"] = "satire_context"
            if snip.get("stance") == "supports_claim":
                snip["stance"] = "satire"

    if role == "fake_news_origin":
        relation = "claim_origin"
        score = max(score, 3)
        if stance in {"supports_claim", "refutes_claim", "refutes_supporting_evidence"}:
            stance = "context_only"
        _add_flag(risk_flags, "fake_news_origin")
        for snip in snippets:
            snip["snippet_type"] = "fake_origin_context"
            if snip.get("stance") in {"supports_claim", "refutes_claim", "refutes_supporting_evidence"}:
                snip["stance"] = "context_only"

    # Visual/provenance roles usually refute support for the claim.
    if role in {"visual_origin", "visual_misattribution", "image_manipulation"}:
        relation = "direct_evidence"
        if stance == "supports_claim":
            stance = "refutes_supporting_evidence"
        if role == "visual_misattribution":
            _add_flag(risk_flags, "image_misattribution")
        if role == "visual_origin":
            _add_flag(risk_flags, "visual_origin_only")
        if role == "image_manipulation":
            _add_flag(risk_flags, "image_manipulation")
        for snip in snippets:
            if snip.get("snippet_type") == "evidence":
                snip["snippet_type"] = "visual_origin"

    # Direct support/refutation roles should be direct_evidence.
    if role == "core_support":
        relation = "direct_evidence"
        if stance in {"context_only", "satire", "not_applicable"}:
            stance = "supports_claim"
    if role == "core_refutation":
        relation = "direct_evidence"
        if stance in {"context_only", "satire", "not_applicable"}:
            stance = "refutes_claim"

    # Context roles are useful_context unless the model found a stronger relation.
    if role in CONTEXT_ROLES:
        if relation == "direct_evidence" and stance in {"supports_claim", "refutes_claim", "refutes_supporting_evidence"}:
            # Keep it direct only if stance is strong; otherwise demote below.
            pass
        else:
            relation = "useful_context"
            stance = "context_only" if stance in {"supports_claim", "refutes_claim", "refutes_supporting_evidence"} else stance
            score = min(score, 3)
            _add_flag(risk_flags, "weak_context_only" if score <= 2 else "background_context")
            for snip in snippets:
                if snip.get("stance") in {"supports_claim", "refutes_claim", "refutes_supporting_evidence"}:
                    snip["stance"] = "context_only"
                snip["snippet_score"] = min(clamp_int(snip.get("snippet_score", 0)), 3)

    # Exact numerical evidence must not be treated as support unless score is high.
    if role == "statistical_evidence" and score < 4:
        _add_flag(risk_flags, "statistical_exactness_needed")
        if stance == "supports_claim":
            stance = "qualifies_claim"

    # Rating caveats and temporal refutations can be direct or useful context, but should be preserved.
    if role in {"rating_caveat", "temporal_refutation"} and score >= 4:
        if relation not in {"direct_evidence", "useful_context"}:
            relation = "direct_evidence"
        if stance == "supports_claim":
            stance = "qualifies_claim"
        _add_flag(risk_flags, role)

    return relation, role, score, stance, snippets, safe_source_fact, risk_flags


def normalize_stage2_output(obj, source, usage=None):
    snippets = normalize_snippets(obj.get("snippets", []))
    risk_flags = normalize_risk_flags(obj.get("risk_flags", []))

    relation = obj.get("source_relation")
    role = obj.get("evidence_role")
    score = clamp_int(obj.get("claim_match_score", 0))
    stance = obj.get("stance_to_claim")
    safe_source_fact = clean_space(obj.get("safe_source_fact", ""))[:700]

    relation, role, score, stance, snippets, safe_source_fact, risk_flags = apply_stage2_guardrails(
        relation=relation,
        role=role,
        score=score,
        stance=stance,
        snippets=snippets,
        safe_source_fact=safe_source_fact,
        risk_flags=risk_flags,
    )

    # Final deterministic keep rules.
    keep_for_writing = bool(obj.get("keep_for_writing"))
    keep_for_coverage = bool(obj.get("keep_for_coverage"))

    if relation not in {"direct_evidence", "claim_origin", "useful_context"}:
        keep_for_writing = False
    if score < 3:
        keep_for_writing = False
    if not snippets:
        keep_for_writing = False

    # Special generation-critical evidence must be available to Stage 3/4 if strong enough.
    if role in SPECIAL_PROMOTION_ROLES and score >= 4 and snippets:
        keep_for_writing = True

    if relation in {"off_topic", "no_content"} or not safe_source_fact:
        keep_for_coverage = False
    elif score >= 3:
        keep_for_coverage = True

    return {
        "id": str(source["id"]),
        "claim": source.get("claim", ""),
        "url": source["url"],
        "file_name": source["file_name"],
        "source_relation": relation,
        "evidence_role": role,
        "claim_match_score": score,
        "stance_to_claim": stance,
        "keep_for_writing": keep_for_writing,
        "keep_for_coverage": keep_for_coverage,
        "why": clean_space(obj.get("why", ""))[:1000],
        "snippets": snippets,
        "safe_source_fact": safe_source_fact,
        "risk_flags": risk_flags,
        "stage1_decision": source.get("stage1_decision"),
        "stage1_confidence": source.get("stage1_confidence"),
        "stage1_reason": source.get("stage1_reason"),
        "stage": "stage2_claim_aware_evidence_extraction_role_risk",
        "model": STAGE2_MODEL,
        "usage": usage or {},
        "run_timestamp_utc": utc_now(),
    }

## 8. API call functions

In [8]:
def make_stage2_config(use_thinking=True):
    kwargs = dict(
        temperature=STAGE2_TEMPERATURE,
        response_mime_type="application/json",
        response_schema=STAGE2_RESPONSE_SCHEMA,
        max_output_tokens=STAGE2_MAX_OUTPUT_TOKENS,
    )

    if use_thinking and STAGE2_THINKING_LEVEL:
        kwargs["thinking_config"] = types.ThinkingConfig(
            thinking_level=STAGE2_THINKING_LEVEL
        )

    return types.GenerateContentConfig(**kwargs)


def stage2_failure_output(source, reason):
    return {
        "id": str(source["id"]),
        "claim": source.get("claim", ""),
        "url": source["url"],
        "file_name": source["file_name"],
        "source_relation": "no_content",
        "evidence_role": "no_content",
        "claim_match_score": 0,
        "stance_to_claim": "unclear",
        "keep_for_writing": False,
        "keep_for_coverage": False,
        "why": f"Stage 2 failed validation/API retries: {reason}",
        "snippets": [],
        "safe_source_fact": "",
        "risk_flags": ["no_readable_content"],
        "stage1_decision": source.get("stage1_decision"),
        "stage1_confidence": source.get("stage1_confidence"),
        "stage1_reason": source.get("stage1_reason"),
        "stage": "stage2_claim_aware_evidence_extraction_role_risk",
        "model": STAGE2_MODEL,
        "usage": {},
        "run_timestamp_utc": utc_now(),
    }


def call_stage2_api(source):
    prompt = build_stage2_prompt(source)
    last_error = None
    use_thinking = True

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=STAGE2_MODEL,
                contents=prompt,
                config=make_stage2_config(use_thinking=use_thinking),
            )

            parsed = extract_json_response(response)
            errors = validate_stage2_model_output(parsed)
            if errors:
                raise ValueError("; ".join(errors))

            return normalize_stage2_output(
                parsed,
                source,
                usage=response_usage_summary(response),
            )

        except Exception as e:
            last_error = repr(e)

            if use_thinking and "thinking" in last_error.lower():
                print("thinking_config rejected; retrying without thinking_config")
                use_thinking = False
                continue

            if "429" in last_error or "RESOURCE_EXHAUSTED" in last_error:
                sleep_s = min(20 * attempt, 90) + random.random() * 5
            else:
                sleep_s = min(2 ** attempt, 20) + random.random()

            print(
                f"Stage 2 attempt {attempt}/{MAX_RETRIES} failed for "
                f"{source['id']} {source['file_name']}: {last_error}. Sleeping {sleep_s:.1f}s"
            )
            time.sleep(sleep_s)

    return stage2_failure_output(source, last_error)

## 9. Smoke test one source

Run this before the full extraction. Inspect the labels carefully, especially `evidence_role`, `stance_to_claim`, `risk_flags`, and `keep_for_writing`.

In [9]:
if TEST_ONE_SOURCE and stage2_inputs:
    test_source = stage2_inputs[0]
    print("Testing one source:")
    print(test_source["id"], test_source["file_name"], test_source["url"])

    test_result = call_stage2_api(test_source)

    print("\nResult:")
    print(json.dumps(test_result, ensure_ascii=False, indent=2)[:5000])
else:
    print("Smoke test skipped. Set TEST_ONE_SOURCE=True and rerun this cell if needed.")

Smoke test skipped. Set TEST_ONE_SOURCE=True and rerun this cell if needed.


## 10. Run Stage 2 extraction with resume support

This cell resumes from any existing local or Drive Stage 2 output. It saves after every completed source by default.

In [10]:
def load_existing_stage2_outputs():
    if STAGE2_OUT.exists():
        print("Loading existing local Stage 2 output:", STAGE2_OUT)
        return load_json(STAGE2_OUT)
    if DRIVE_STAGE2_OUT.exists():
        print("Loading existing Drive Stage 2 output:", DRIVE_STAGE2_OUT)
        existing = load_json(DRIVE_STAGE2_OUT)
        save_json(STAGE2_OUT, existing)
        return existing
    return []


def sorted_outputs_from_map(output_by_key):
    return sorted(
        output_by_key.values(),
        key=lambda x: (
            str(x.get("id")),
            str(x.get("file_name")),
            str(x.get("url")),
        ),
    )


def save_stage2_progress(output_by_key):
    rows = sorted_outputs_from_map(output_by_key)
    save_json(STAGE2_OUT, rows)
    copy_to_drive(STAGE2_OUT, DRIVE_STAGE2_OUT)
    return rows


existing = load_existing_stage2_outputs()
output_by_key = {source_key(x): x for x in existing}

todo = [s for s in stage2_inputs if source_key(s) not in output_by_key]
if MAX_API_CALLS is not None:
    todo = todo[:MAX_API_CALLS]

print("Existing Stage 2 outputs:", len(existing))
print("Remaining Stage 2 API calls in this run:", len(todo))
print("MAX_WORKERS:", MAX_WORKERS)

completed_this_run = 0

if not todo:
    print("Nothing to do.")
elif MAX_WORKERS <= 1:
    for idx, source in enumerate(todo, start=1):
        print(
            f"\nCalling {idx}/{len(todo)} | "
            f"{source['id']} {source['file_name']} | "
            f"{source['url'][:120]}"
        )
        result = call_stage2_api(source)
        output_by_key[source_key(source)] = result
        completed_this_run += 1

        if completed_this_run % SAVE_AFTER_EVERY_COMPLETED == 0:
            rows = save_stage2_progress(output_by_key)
            usage = result.get("usage") or {}
            print(
                f"Saved {len(rows)}/{len(stage2_inputs)} | "
                f"{result['id']} {result['file_name']} -> "
                f"{result['source_relation']}/{result['evidence_role']} "
                f"score={result['claim_match_score']} "
                f"keep_write={result['keep_for_writing']} "
                f"tokens={usage.get('total_token_count')} thoughts={usage.get('thoughts_token_count')}"
            )
else:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_source = {executor.submit(call_stage2_api, source): source for source in todo}

        for future in as_completed(future_to_source):
            source = future_to_source[future]
            try:
                result = future.result()
            except Exception as e:
                result = stage2_failure_output(source, repr(e))

            output_by_key[source_key(source)] = result
            completed_this_run += 1

            if completed_this_run % SAVE_AFTER_EVERY_COMPLETED == 0:
                rows = save_stage2_progress(output_by_key)
                usage = result.get("usage") or {}
                print(
                    f"Saved {len(rows)}/{len(stage2_inputs)} | "
                    f"completed_this_run={completed_this_run}/{len(todo)} | "
                    f"{result['id']} {result['file_name']} -> "
                    f"{result['source_relation']}/{result['evidence_role']} "
                    f"score={result['claim_match_score']} "
                    f"keep_write={result['keep_for_writing']} "
                    f"tokens={usage.get('total_token_count')} thoughts={usage.get('thoughts_token_count')}"
                )

rows = save_stage2_progress(output_by_key)
print("\nFinal Stage 2 rows:", len(rows))
print("Stage 2 local output:", STAGE2_OUT)
print("Stage 2 Drive output:", DRIVE_STAGE2_OUT)

Existing Stage 2 outputs: 0
Remaining Stage 2 API calls in this run: 5741
MAX_WORKERS: 8
Saved 20/5741 | completed_this_run=20/5741 | 30355_exclaim 30355_6.json -> claim_origin/claim_origin score=3 keep_write=True tokens=3148 thoughts=609
Saved 40/5741 | completed_this_run=40/5741 | 30361_exclaim 30361_1.json -> claim_origin/claim_origin score=3 keep_write=True tokens=3005 thoughts=429
Saved 60/5741 | completed_this_run=60/5741 | 30366_exclaim 30366_2.json -> off_topic/off_topic score=0 keep_write=False tokens=7735 thoughts=833
Saved 80/5741 | completed_this_run=80/5741 | 30369_exclaim 30369_9.json -> direct_evidence/core_refutation score=5 keep_write=True tokens=6077 thoughts=742
Saved 100/5741 | completed_this_run=100/5741 | 30414_exclaim 30414_3.json -> useful_context/background_context score=3 keep_write=True tokens=8615 thoughts=1339
Saved 120/5741 | completed_this_run=120/5741 | 30440_exclaim 30440_2.json -> useful_context/background_context score=3 keep_write=True tokens=5252 th

## 11. Inspect one claim ID

In [17]:
stage2_outputs = load_json(STAGE2_OUT)
items = [x for x in stage2_outputs if str(x.get("id")) == str(INSPECT_ID)]

INSPECT_ID = '118_ambigsnopes'

print(f"Stage 2 output file: {STAGE2_OUT}")
print(f"Claim ID: {INSPECT_ID}")
print(f"Number of source outputs: {len(items)}")
print("=" * 100)

for i, item in enumerate(items, start=1):
    print(f"\nSOURCE {i}")
    print("-" * 100)
    print("file_name:", item.get("file_name"))
    print("url:", item.get("url"))
    print("relation:", item.get("source_relation"))
    print("role:", item.get("evidence_role"))
    print("score:", item.get("claim_match_score"))
    print("stance:", item.get("stance_to_claim"))
    print("risk_flags:", item.get("risk_flags"))
    print("keep_for_writing:", item.get("keep_for_writing"))
    print("keep_for_coverage:", item.get("keep_for_coverage"))
    print("why:", item.get("why"))
    print("safe_source_fact:", item.get("safe_source_fact"))
    print("usage:", item.get("usage"))
    print("snippets:")

    for snip in item.get("snippets", []):
        print(
            "  -",
            snip.get("snippet_type"),
            "|",
            snip.get("stance"),
            "|",
            snip.get("snippet_score"),
        )
        print("   ", snip.get("text"))

Stage 2 output file: /content/outputs/stage2/stage2_extract_test_dataset.json
Claim ID: 118_ambigsnopes
Number of source outputs: 4

SOURCE 1
----------------------------------------------------------------------------------------------------
file_name: 190.txt
url: https://www.vogue.com/13335624/vogue-podcast
relation: no_content
role: no_content
score: 0
stance: unclear
risk_flags: ['no_readable_content']
keep_for_writing: False
keep_for_coverage: False
why: The source text provided is empty, offering no information regarding the claim about Kim Kardashian's trip to Paris for cheesecake.
safe_source_fact: 
usage: {'prompt_token_count': 2144, 'candidates_token_count': 106, 'thoughts_token_count': 1887, 'total_token_count': 4137}
snippets:

SOURCE 2
----------------------------------------------------------------------------------------------------
file_name: 435.txt
url: https://www.standard.co.uk/showbiz/celebrity-news/kim-kardashian-reveals-she-flew-to-paris-for-a-slice-of-cheesecak

## 12. Output summary by claim

Use this to catch claims with too few writing/coverage sources before moving to Stage 3.

In [12]:
stage2_outputs = load_json(STAGE2_OUT)

by_claim = defaultdict(list)
for x in stage2_outputs:
    by_claim[str(x.get("id"))].append(x)

for claim_id in sorted(by_claim.keys()):
    rows = by_claim[claim_id]
    rel_counts = Counter(x.get("source_relation") for x in rows)
    role_counts = Counter(x.get("evidence_role") for x in rows)
    stance_counts = Counter(x.get("stance_to_claim") for x in rows)
    kept = sum(1 for x in rows if x.get("keep_for_writing"))
    coverage = sum(1 for x in rows if x.get("keep_for_coverage"))
    flags = Counter(flag.split(":", 1)[0] for x in rows for flag in x.get("risk_flags", []))

    print(
        claim_id,
        "| total:", len(rows),
        "| keep_for_writing:", kept,
        "| keep_for_coverage:", coverage,
        "| relations:", dict(rel_counts),
        "| roles:", dict(role_counts),
        "| stances:", dict(stance_counts),
        "| risk_flags:", dict(flags),
    )

118_ambigsnopes | total: 4 | keep_for_writing: 0 | keep_for_coverage: 0 | relations: {'no_content': 4} | roles: {'no_content': 4} | stances: {'unclear': 4} | risk_flags: {'no_readable_content': 4}
122_ambigsnopes | total: 2 | keep_for_writing: 0 | keep_for_coverage: 0 | relations: {'no_content': 2} | roles: {'no_content': 2} | stances: {'unclear': 2} | risk_flags: {'no_readable_content': 2}
128_ambigsnopes | total: 2 | keep_for_writing: 0 | keep_for_coverage: 0 | relations: {'no_content': 2} | roles: {'no_content': 2} | stances: {'unclear': 2} | risk_flags: {'no_readable_content': 2}
129_ambigsnopes | total: 2 | keep_for_writing: 0 | keep_for_coverage: 0 | relations: {'no_content': 2} | roles: {'no_content': 2} | stances: {'unclear': 2} | risk_flags: {'no_readable_content': 2}
12_ambigsnopes | total: 8 | keep_for_writing: 0 | keep_for_coverage: 0 | relations: {'no_content': 8} | roles: {'no_content': 8} | stances: {'unclear': 8} | risk_flags: {'no_readable_content': 8, 'no_content': 1}

## 13. Optional: show Stage 3 handoff candidates

This does not replace Stage 3. It only previews the sources that Stage 3 will likely promote or consider.

In [13]:
PREVIEW_ID = INSPECT_ID
stage2_outputs = load_json(STAGE2_OUT)
items = [x for x in stage2_outputs if str(x.get("id")) == str(PREVIEW_ID)]

priority_role_order = {
    "core_refutation": 1,
    "core_support": 1,
    "visual_misattribution": 2,
    "visual_origin": 2,
    "image_manipulation": 2,
    "satirical_origin": 2,
    "fake_news_origin": 2,
    "temporal_refutation": 3,
    "rating_caveat": 3,
    "statistical_evidence": 3,
    "claim_origin": 4,
    "legal_or_definition_context": 5,
    "scientific_context": 5,
    "background_context": 6,
}

candidates = sorted(
    [x for x in items if x.get("keep_for_writing") or x.get("keep_for_coverage")],
    key=lambda x: (
        priority_role_order.get(x.get("evidence_role"), 99),
        -int(x.get("claim_match_score", 0)),
        x.get("file_name", ""),
    ),
)

print(f"Likely Stage 3 candidates for claim {PREVIEW_ID}: {len(candidates)}")
for x in candidates[:10]:
    print("-", x.get("file_name"), "|", x.get("evidence_role"), "| score", x.get("claim_match_score"), "|", x.get("url"))
    print("  fact:", x.get("safe_source_fact"))
    if x.get("risk_flags"):
        print("  flags:", x.get("risk_flags"))

Likely Stage 3 candidates for claim 30350: 0


In [14]:
stage2_outputs = load_json(STAGE2_OUT)

ids = sorted({str(x.get("id")) for x in stage2_outputs})

print("Number of unique claim IDs in Stage 2:", len(ids))
print("First 30 IDs:")
print(ids[:30])
print("\nLast 30 IDs:")
print(ids[-30:])

Number of unique claim IDs in Stage 2: 1022
First 30 IDs:
['118_ambigsnopes', '122_ambigsnopes', '128_ambigsnopes', '129_ambigsnopes', '12_ambigsnopes', '145_ambigsnopes', '146_ambigsnopes', '149_ambigsnopes', '150_ambigsnopes', '151_ambigsnopes', '153_ambigsnopes', '154_ambigsnopes', '158_ambigsnopes', '168_ambigsnopes', '171_ambigsnopes', '179_ambigsnopes', '180_ambigsnopes', '187_ambigsnopes', '188_ambigsnopes', '196_ambigsnopes', '197_ambigsnopes', '198_ambigsnopes', '200_ambigsnopes', '207_ambigsnopes', '208_ambigsnopes', '28_ambigsnopes', '2_ambigsnopes', '30354_exclaim', '30355_exclaim', '30356_exclaim']

Last 30 IDs:
['33687_exclaim', '33690_exclaim', '33694_exclaim', '33696_exclaim', '33700_exclaim', '33701_exclaim', '33704_exclaim', '33706_exclaim', '33713_exclaim', '33716_exclaim', '33718_exclaim', '33720_exclaim', '36_ambigsnopes', '38_ambigsnopes', '42_ambigsnopes', '50_ambigsnopes', '51_ambigsnopes', '54_ambigsnopes', '55_ambigsnopes', '56_ambigsnopes', '60_ambigsnopes', 

In [18]:
DEBUG_ID = "118_ambigsnopes"

# Load Stage 0 flattened sources, because Stage 2 may have trimmed text.
flattened_sources = load_json(STAGE0_IN)

debug_sources = [
    x for x in flattened_sources
    if str(x.get("id")) == str(DEBUG_ID)
]

print("DEBUG_ID:", DEBUG_ID)
print("Number of Stage 0 source records:", len(debug_sources))
print("=" * 120)

for i, src in enumerate(debug_sources, start=1):
    print(f"\nSOURCE {i}")
    print("-" * 120)
    print("file_name:", src.get("file_name"))
    print("url:", src.get("url"))
    print("article_path:", src.get("article_path"))
    print("load_error:", src.get("load_error"))
    print("source_text_original_chars:", src.get("source_text_original_chars"))
    print("source_text_packed_chars:", src.get("source_text_packed_chars"))
    print("source_text_was_packed:", src.get("source_text_was_packed"))

    text = src.get("source_text") or ""
    print("\nSOURCE TEXT PREVIEW:")
    print(repr(text[:3000]))

    if len(text) > 3000:
        print("\n... [middle omitted in preview] ...")
        print("\nSOURCE TEXT TAIL:")
        print(repr(text[-1500:]))

DEBUG_ID: 118_ambigsnopes
Number of Stage 0 source records: 14

SOURCE 1
------------------------------------------------------------------------------------------------------------------------
file_name: 33.txt
url: https://hollywoodlife.com/pics/kim-kardashian-paris-fashion-week-2015-outfits-photos
article_path: /content/data/test_dataset/evidence_docs/33.txt
load_error: JSONDecodeError('Expecting value: line 15 column 1 (char 14)')
source_text_original_chars: 0
source_text_packed_chars: 0
source_text_was_packed: False

SOURCE TEXT PREVIEW:
''

SOURCE 2
------------------------------------------------------------------------------------------------------------------------
file_name: 311.txt
url: https://www.vogue.com.au/celebrity/news/there-are-still-things-you-dont-know-about-kim-kardashian-west/news-story/ca74d2b8098589ce8676f34b7aef4475
article_path: /content/data/test_dataset/evidence_docs/311.txt
load_error: JSONDecodeError('Expecting value: line 1 column 2 (char 1)')
source_tex

In [19]:
DEBUG_ID = "118_ambigsnopes"

flattened_sources = load_json(STAGE0_IN)
debug_sources = [
    x for x in flattened_sources
    if str(x.get("id")) == str(DEBUG_ID)
]

for src in debug_sources:
    path = Path(src.get("article_path", ""))
    print("\n" + "=" * 120)
    print("file_name:", src.get("file_name"))
    print("path:", path)
    print("exists:", path.exists())

    if path.exists():
        raw = path.read_text(encoding="utf-8", errors="replace")
        print("raw file chars:", len(raw))
        print("raw preview:")
        print(repr(raw[:3000]))


file_name: 33.txt
path: /content/data/test_dataset/evidence_docs/33.txt
exists: False

file_name: 311.txt
path: /content/data/test_dataset/evidence_docs/311.txt
exists: False

file_name: 134.txt
path: /content/data/test_dataset/evidence_docs/134.txt
exists: False

file_name: 276.txt
path: /content/data/test_dataset/evidence_docs/276.txt
exists: False

file_name: 985.txt
path: /content/data/test_dataset/evidence_docs/985.txt
exists: False

file_name: 439.txt
path: /content/data/test_dataset/evidence_docs/439.txt
exists: False

file_name: 993.txt
path: /content/data/test_dataset/evidence_docs/993.txt
exists: False

file_name: 861.txt
path: /content/data/test_dataset/evidence_docs/861.txt
exists: False

file_name: 435.txt
path: /content/data/test_dataset/evidence_docs/435.txt
exists: False

file_name: 797.txt
path: /content/data/test_dataset/evidence_docs/797.txt
exists: False

file_name: 656.txt
path: /content/data/test_dataset/evidence_docs/656.txt
exists: False

file_name: 190.txt
pat